<a href="https://colab.research.google.com/github/simon-mellergaard/GAI-with-LLMs/blob/main/Project%20codes/Assignment05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 5

> With point of departure in Chapter 6 in Hands-on Generative AI with Transformers and Diffusion Models, carefully fine-tune mistralai/Mistral-7B-v0.3 while manipulating key hyperparameters. For your best model, present the training-validation plot and completions for the prompts below:

``` python
prompts = [
    """What is the capital of Germany? Explain why thats the case and if it \
    was different in the past?""",
    "Write a Python function to calculate the factorial of a number.",
    """A rectangular garden has a length of 25 feet and a width of 15 feet. \
    If you want to build a fence around the entire garden, how many feet of \
    fencing will you need?""",
    """What is the difference between a fruit and a vegetable? Give examples \
    of each.""",
]
```

## Setup

In [21]:
!pip install trl
!pip install bitsandbytes
# !pip install genaibook

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.3 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of opencv-contrib-python to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of opencv-python-headless to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.4/290.4 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1

In [5]:
# Libraries
import os
import torch
import pandas as pd
import matplotlib.pyplot as plt

# Functions
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer
from datasets import load_dataset
from google.colab import userdata
from huggingface_hub import login as login_hf
from wandb import login as login_wandb
from peft import LoraConfig
from transformers import AutoTokenizer
from transformers import pipeline

In [6]:
# Logging in to Hugging Face and wandb
os.environ['HF_TOKEN'] = userdata.get('HF')
os.environ['WANDB_TOKEN'] = userdata.get('wandb')
os.environ['HF_USER'] = userdata.get('HF_USER')
login_hf(os.environ['HF_TOKEN'])
login_wandb(key = os.environ['WANDB_TOKEN'])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: simonmellergaard (simonmellergaard-aarhus-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [9]:
# Setting up the device (GPU)
device = "cuda" if torch.cuda.is_available() else "cpu"

## Finetuning mistral with adapters

In [17]:
!pip list | grep bitsandbytes

bitsandbytes                          0.48.0


The model used is the [Mistral-7B model](https://huggingface.co/mistralai/Mistral-7B-v0.3), which has more than 7 billion parameters.

In [4]:
quantization_config = BitsAndBytesConfig(load_in_4bit=True)

model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-v0.3",
    quantization_config=quantization_config,
    device_map="auto",
    attn_implementation="flash_attention_2" # Flash attention to train faster
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.55G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

[Dataset](https://huggingface.co/datasets/timdettmers/openassistant-guanaco)

In [8]:
dataset = load_dataset("timdettmers/openassistant-guanaco", split="train")
dataset_eval = load_dataset("timdettmers/openassistant-guanaco", split="test")

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

Repo card metadata block was not found. Setting CardData to empty.


In [11]:
sft_config = SFTConfig(
    "mistral-finetuned",
    push_to_hub=True,
    per_device_train_batch_size=4, # Might need to reduce
    weight_decay=0.1,
    lr_scheduler_type="cosine",
    learning_rate=5e-4,
    num_train_epochs=2,
    eval_strategy="epoch",
    # eval_steps=200,
    logging_steps=10,
    gradient_checkpointing=True,
    max_length=512, # Changed from max_seq_length
    dataset_text_field="text",
    packing=True,
)


trainer = SFTTrainer(
    model,
    args=sft_config,
    train_dataset=dataset.select(range(300)),
    eval_dataset=dataset_eval.select(range(100)),
    peft_config=peft_config,
)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:929: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
1,1.275300,1.103132,1.147905,103388.000000,0.726500


In [ ]:
trainer.push_to_hub()

## Plotting the training loss

In [ ]:
def plot_loss(trainer, title='Training and Validation Loss over Epochs'):
    # Creating dataframe for plotting. Load pandas and matplotlib
    history = trainer.state.log_history[:-1]
    results = pd.DataFrame(columns=pd.DataFrame(history).columns)
    # The results are stored in alternating order in the results, so it is
    # iterated through every second instance
    for i in range(len(history)):
        if i % 2 == 0:
            new_row = pd.DataFrame(history[i], index=[0])
            results = pd.concat([results, new_row], ignore_index=True)
        else:
            # The bottom most row is inserted the specific columns
            results.loc[len(results) - 1, history[i].keys()] = history[i]
    # Plotting the results
    plt.figure(figsize=(7, 5))
    plt.plot(results['epoch'], results['loss'], marker='o', label='Training Loss', color = 'blue')
    plt.plot(results['epoch'], results['eval_loss'], marker='o', label='Validation Loss', color = 'orange')
    plt.title(title)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:
plot_loss(trainer, 'Fine-tuning mistral-7B')

## Loading the model and adapter

In [ ]:
# Loading the base model
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.3")
model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-v0.3",
    dtype=torch.float16,
    device_map="auto",
)

# Loading the fine-tuned adapter.
model.load_adapter("simon-mellergaard/mistral-finetuned")

Using a pipeline to test the 4 different queries

In [ ]:
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
pipe("""### Human: What is the capital of Germany? Explain why thats the case \
    and if it was different in the past?### Assistant:""", max_new_tokens=100)
pipe("""### Human: Write a Python function to calculate the factorial of a \
    number.### Assistant:""", max_new_tokens=100)
pipe("""### Human: A rectangular garden has a length of 25 feet and a width of \
    15 feet. If you want to build a fence around the entire garden, how many \
    feet of fencing will you need?### Assistant:""", max_new_tokens=100)
pipe("""### Human: What is the difference between a fruit and a vegetable? Give \
     examples of each.### Assistant:""", max_new_tokens=100)

Using the chat template

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/Mistral-7B-v0.3")
chat = [
    {"role": "user", "content": """What is the capital of Germany? Explain why thats the case \
    and if it was different in the past?"""}
]
tokenized_chat = tokenizer.apply_chat_template(chat, tokenize=False)
outputs = model.generate(tokenized_chat, max_new_tokens=128)
print(tokenizer.decode(outputs[0]))

Using a function

In [ ]:
def generate_text(prompt):
    pipe(f"### Human: {prompt}### Assistant:", max_new_tokens=100)

In [ ]:
generate_text("""What is the capital of Germany? Explain why thats the case and
              if it was different in the past?""")
generate_text("Write a Python function to calculate the factorial of a number.")
generate_text("""A rectangular garden has a length of 25 feet and a width of 15
              feet. If you want to build a fence around the entire garden, how
              many feet of fencing will you need?""")
generate_text("""What is the difference between a fruit and a vegetable? Give
              examples of each.""")